In [ ]:
from IPython.display import HTML

HTML("""
<style>
/* Force all output panels to stack vertically */
.fft-row {
    display: flex !important;
    flex-direction: column !important;
    justify-content: center !important;
    align-items: center !important;
}

/* Ensure panels take full width */
.fft-panel {
    width: 100% !important;
    margin-bottom: 20px;
}
</style>
""")

import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------------
# Helpers
# -------------------------------
def get_uploaded_file_content(upload_widget):
    """Safely extract bytes from FileUpload widget."""
    if not upload_widget.value:
        return None
    val = upload_widget.value
    if isinstance(val, dict):
        return list(val.values())[0]["content"]
    if isinstance(val, tuple):
        return val[0]["content"]
    return None

def normalize_image(img):
    """Normalize array to uint8 for display."""
    img = img - np.min(img)
    if np.max(img) > 0:
        img = img / np.max(img)
    return (img * 255).astype(np.uint8)

def resize_if_needed(pil_img, max_size=512):
    """Resize using thumbnail() while keeping aspect ratio."""
    if pil_img.width > max_size or pil_img.height > max_size:
        pil_img = pil_img.copy()
        pil_img.thumbnail((max_size, max_size))
    return pil_img

def to_grayscale_array(pil_img):
    """Return grayscale numpy array."""
    if pil_img.mode != "L":
        pil_img = pil_img.convert("L")
    return np.array(pil_img, dtype=np.float32)

def circular_mask(shape, radius, filter_type):
    """Return mask for None / Low-pass / High-pass."""
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    Y, X = np.ogrid[:rows, :cols]
    dist = np.sqrt((X - ccol)**2 + (Y - crow)**2)
    if filter_type == 'Low-pass':
        mask = dist <= radius
    elif filter_type == 'High-pass':
        mask = dist >= radius
    else:
        mask = np.ones_like(dist, dtype=bool)
    return mask.astype(float)

# -------------------------------
# Widgets
# -------------------------------
title_html = widgets.HTML(
    "<h2 style='margin-bottom:8px;'>Interactive Fourier Filtering Explorer</h2>"
)

subtitle_html = widgets.HTML(
    "<p style='margin-top:-10px;margin-bottom:12px;'>"
    "Upload an image, select a filter type, and visualize how the frequency-domain mask affects the image."
    "</p>"
)

upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="📁 Upload Image",
    style={'description_width': '80px'}
)

filter_type = widgets.Dropdown(
    options=['None', 'Low-pass', 'High-pass'],
    value='Low-pass',
    description='Filter:',
)

radius_slider = widgets.IntSlider(
    value=40, min=1, max=400, step=1,
    description="Radius:", continuous_update=False
)

cmap_selector = widgets.Dropdown(
    options=['inferno', 'magma', 'viridis', 'gray'],
    value='magma',
    description='FFT colormap:'
)

out_status = widgets.Output()
out_original = widgets.Output()
out_fft = widgets.Output()
out_filtered = widgets.Output()

def compute_radius_from_click(event, arr_shape):
    """Compute radial distance of click from FFT center."""
    if event.xdata is None or event.ydata is None:
        return None
    rows, cols = arr_shape
    crow, ccol = rows // 2, cols // 2
    x, y = event.xdata, event.ydata
    r = np.sqrt((x - ccol)**2 + (y - crow)**2)
    return int(r)


# -------------------------------
# Processing & Display
# -------------------------------
def process_and_display(change=None):

    out_status.clear_output(wait=True)
    out_original.clear_output(wait=True)
    out_fft.clear_output(wait=True)
    out_filtered.clear_output(wait=True)

    content = get_uploaded_file_content(upload)
    if content is None:
        with out_status:
            print("Waiting for image upload...")
        return

    try:
        with out_status:
            print("Processing image...")

        pil_img = Image.open(io.BytesIO(content))
        pil_img = resize_if_needed(pil_img)
        gray_arr = to_grayscale_array(pil_img)

        # FFT
        F = np.fft.fft2(gray_arr)
        Fshift = np.fft.fftshift(F)
        magnitude = np.abs(Fshift)
        display_magnitude = np.log1p(magnitude)  # auto normalization

        # Mask
        rad = radius_slider.value
        mask = circular_mask(gray_arr.shape, rad, filter_type.value)
        Fshift_filtered = Fshift * mask

        # Inverse FFT
        F_ishift = np.fft.ifftshift(Fshift_filtered)
        img_filtered = np.abs(np.fft.ifft2(F_ishift))
        img_filtered_norm = normalize_image(img_filtered)

        # ORIGINAL
        with out_original:
            plt.figure(figsize=(3.5,3.5))
            plt.imshow(pil_img if pil_img.mode in ("RGB","RGBA") else gray_arr, cmap="gray")
            plt.title("Original Image")
            plt.axis('off')
            plt.show()

        # ---------------- Display FFT magnitude + mask overlay + CLICK INTERACTION ----------------
        with out_fft:
            fig, ax = plt.subplots(figsize=(3.5, 3.5))

            ax.imshow(display_magnitude, cmap=cmap_selector.value)

            # Draw mask overlay
            if filter_type.value != "None":
                overlay = np.zeros((*mask.shape, 4))
                overlay[mask == 1] = [1, 1, 0, 0.35]  # yellow transparent
                ax.imshow(overlay)

            ax.set_title(f"FFT Magnitude (Tap to change radius)", fontsize=10)
            ax.axis('off')

            # Click handler
            def onclick(event):
                r = compute_radius_from_click(event, gray_arr.shape)
                if r is not None:
                    radius_slider.value = r  # triggers full recompute

            # Connect click listener
            cid = fig.canvas.mpl_connect("button_press_event", onclick)

            plt.tight_layout()
            plt.show()

        # FILTERED IMAGE
        with out_filtered:
            plt.figure(figsize=(3.5,3.5))
            plt.imshow(img_filtered_norm, cmap="gray")
            plt.title(f"Filtered Output ({filter_type.value})")
            plt.axis('off')
            plt.show()

    except Exception as e:
        with out_status:
            print("Error:", e)

# -------------------------------
# Observers
# -------------------------------
upload.observe(process_and_display, names='value')
filter_type.observe(process_and_display, names='value')
radius_slider.observe(process_and_display, names='value')
cmap_selector.observe(process_and_display, names='value')

# -------------------------------
# Layout (clean & professional)
# -------------------------------
control_box = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    widgets.HBox([upload, filter_type]),
    widgets.HBox([radius_slider, cmap_selector]),
    out_status
])

output_box = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Original</b>"), out_original]),
    widgets.VBox([widgets.HTML("<b>FFT + Mask</b>"), out_fft]),
    widgets.VBox([widgets.HTML("<b>Filtered Output</b>"), out_filtered]),
])

display(widgets.VBox([
    title_html,
    subtitle_html,
    control_box,
    widgets.HTML("<hr>"),
    output_box
]))
